In [9]:
GT_ROOT = "/home/liwa/data/datasets/yeastcity/gtFine/test/10001_10130/"
PRED_ROOT = "./predicted_v3model/"

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from tqdm import trange
from skimage.io import imread
# ============================================================
# Class colors
# class ID: 1–5
# ============================================================
CLASS_COLORS = {
    0: (1.00, 1.00, 1.00),  # background: white
    1: (1.00, 0.10, 0.10),  # class 1: red
    2: (0.10, 0.80, 0.10),  # class 2: green
    3: (0.10, 0.35, 1.00),  # class 3: blue
    4: (1.00, 0.65, 0.05),  # class 4: orange
    5: (0.65, 0.20, 0.90),  # class 5: purple
    6: (0.00, 0.75, 0.35),  # 5 Over-segmentation - green
    7: (0.85, 0.00, 0.60),  # 6 Under-segmentation - magenta
    8: (0.85, 0.15, 0.15),  # 7 Boundary error - red
    9: (0.55, 0.25, 0.85),  # 8 Split error - purple
    10: (0.00, 0.75, 0.75),  # 9 Merge error - cyan
    11: (0.75, 0.60, 0.00),  #10 Low confidence - mustard
    12: (0.60, 0.30, 0.10),  #11 Unknown / Other - brown
}


def extract_class_map(instance_mask):
    """
    Convert encoded instance mask into class map.

    Encoding:
        pixel value = class_id * 1000 + instance_id

    Background:
        0
    """
    instance_mask = np.asarray(instance_mask).astype(np.int64)

    class_map = instance_mask // 1000
    class_map[instance_mask == 0] = 0

    return class_map


def class_map_to_rgb(class_map, class_colors=CLASS_COLORS):
    """
    Convert class IDs into an RGB image.
    """
    class_map = np.asarray(class_map)

    rgb = np.ones((*class_map.shape, 3), dtype=np.float32)

    for class_id, color in class_colors.items():
        rgb[class_map == class_id] = color

    return rgb


def create_error_map(
    pred_mask,
    gt_mask,
    show_class_mismatch=True
):
    """
    Create a pixel-level segmentation error map.

    Error labels:
        0 = background / not evaluated
        1 = correct foreground prediction
        2 = false positive
        3 = false negative
        4 = class mismatch
    """
    pred_mask = np.asarray(pred_mask).astype(np.int64)
    gt_mask = np.asarray(gt_mask).astype(np.int64)

    if pred_mask.shape != gt_mask.shape:
        raise ValueError(
            f"Shape mismatch: prediction {pred_mask.shape}, "
            f"GT {gt_mask.shape}"
        )

    pred_fg = pred_mask > 0
    gt_fg = gt_mask > 0

    pred_class = extract_class_map(pred_mask)
    gt_class = extract_class_map(gt_mask)

    error_map = np.zeros(gt_mask.shape, dtype=np.uint8)

    # Correct foreground
    correct_fg = pred_fg & gt_fg
    error_map[correct_fg] = 1

    # False positive
    false_positive = pred_fg & ~gt_fg
    error_map[false_positive] = 2

    # False negative
    false_negative = ~pred_fg & gt_fg
    error_map[false_negative] = 3

    # Both are foreground, but class is incorrect
    if show_class_mismatch:
        class_mismatch = (
            pred_fg
            & gt_fg
            & (pred_class != gt_class)
        )
        error_map[class_mismatch] = 4

    return error_map


def visualize_prediction_gt_error(
    raw_image,
    pred_mask,
    gt_mask,
    show_class_mismatch=True,
    save_path="raw_prediction_gt_error.png",
    dpi=300
):
    """
    Save raw image, prediction, ground truth and error map
    as four subplots in one horizontal row.

    The figure is saved as PNG and is not displayed in JupyterLab.

    Parameters
    ----------
    raw_image : (H, W) or (H, W, 3)
        Raw grayscale or RGB image.

    pred_mask : (H, W)
        Prediction encoded as:
        class_id * 1000 + instance_id

    gt_mask : (H, W)
        Ground truth encoded as:
        class_id * 1000 + instance_id

    show_class_mismatch : bool
        Whether to show class mismatch as a separate error type.

    save_path : str
        PNG output path.

    dpi : int
        Saving resolution.

    Returns
    -------
    error_map : (H, W) uint8 array
    """
    raw_image = np.asarray(raw_image)

    if raw_image.shape[:2] != np.asarray(gt_mask).shape:
        raise ValueError(
            f"Raw image shape {raw_image.shape[:2]} does not match "
            f"mask shape {np.asarray(gt_mask).shape}"
        )

    # Ensure PNG extension
    if not save_path.lower().endswith(".png"):
        save_path += ".png"

    pred_class = extract_class_map(pred_mask)
    gt_class = extract_class_map(gt_mask)

    pred_rgb = class_map_to_rgb(pred_class)
    gt_rgb = class_map_to_rgb(gt_class)

    error_map = create_error_map(
        pred_mask,
        gt_mask,
        show_class_mismatch=show_class_mismatch
    )

    if show_class_mismatch:
        error_colors = [
            (0.00, 0.00, 0.00),  # 0: background
            (0.00, 0.00, 0.00),  # 1: correct
            (1.00, 0.55, 0.00),  # 2: false positive
            (0.10, 0.45, 1.00),  # 3: false negative
            (0.30, 0.30, 0.30),  # 4: class mismatch

        ]

        error_labels = [
            "Background",
            "Correct",
            "False positive",
            "False negative",
            "Class mismatch",
        ]

    else:
        error_colors = [
            (0.00, 0.00, 0.00),  # 0: background
            (0.00, 0.00, 0.00),  # 1: correct
            (1.00, 0.55, 0.00),  # 2: false positive
            (0.10, 0.45, 1.00),  # 3: false negative
        ]

        error_labels = [
            "Background",
            "Correct",
            "False positive",
            "False negative",
        ]

    error_cmap = ListedColormap(error_colors)

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(16, 4),
        constrained_layout=True
    )

    # --------------------------------------------------------
    # 1. Raw image
    # --------------------------------------------------------
    if raw_image.ndim == 2:
        axes[0].imshow(raw_image, cmap="gray")
    elif raw_image.ndim == 3:
        axes[0].imshow(raw_image)
    else:
        raise ValueError(
            "raw_image must have shape (H, W) or (H, W, 3)."
        )

    axes[0].set_title("Raw image")

    # --------------------------------------------------------
    # 2. Prediction
    # --------------------------------------------------------
    axes[1].imshow(
        pred_rgb,
        interpolation="nearest"
    )
    axes[1].set_title("Prediction")

    # --------------------------------------------------------
    # 3. Ground truth
    # --------------------------------------------------------
    axes[2].imshow(
        gt_rgb,
        interpolation="nearest"
    )
    axes[2].set_title("Ground truth")

    # --------------------------------------------------------
    # 4. Error map
    # --------------------------------------------------------
    axes[3].imshow(
        error_map,
        cmap=error_cmap,
        vmin=0,
        vmax=len(error_colors) - 1,
        interpolation="nearest"
    )
    axes[3].set_title("Error map")

    for ax in axes:
        ax.axis("off")

    # Error legend
    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="s",
            linestyle="none",
            markerfacecolor=color,
            markeredgecolor="black",
            markeredgewidth=0.5,
            markersize=9,
            label=label
        )
        for color, label in zip(
            error_colors[1:],
            error_labels[1:]
        )
    ]

    axes[3].legend(
        handles=handles,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        frameon=False
    )

    # Save only; do not display in JupyterLab
    fig.savefig(
        save_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    return error_map

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


# ============================================================
# Class colors
# class ID: 1–5
# ============================================================
CLASS_COLORS = {
    0: (1.00, 1.00, 1.00),  # background: white
    1: (1.00, 0.10, 0.10),  # class 1: red
    2: (0.10, 0.80, 0.10),  # class 2: green
    3: (0.10, 0.35, 1.00),  # class 3: blue
    4: (1.00, 0.65, 0.05),  # class 4: orange
    5: (0.65, 0.20, 0.90),  # class 5: purple
    6: (0.00, 0.75, 0.35),  # 5 Over-segmentation - green
    7: (0.85, 0.00, 0.60),  # 6 Under-segmentation - magenta
    8: (0.85, 0.15, 0.15),  # 7 Boundary error - red
    9: (0.55, 0.25, 0.85),  # 8 Split error - purple
    10: (0.00, 0.75, 0.75),  # 9 Merge error - cyan
    11: (0.75, 0.60, 0.00),  #10 Low confidence - mustard
    12: (0.60, 0.30, 0.10),  #11 Unknown / Other - brown
}


def extract_class_map(instance_mask):
    """
    Convert encoded instance mask into class map.

    Encoding:
        pixel value = class_id * 1000 + instance_id
    """
    instance_mask = np.asarray(instance_mask).astype(np.int64)

    class_map = instance_mask // 1000
    class_map[instance_mask == 0] = 0

    return class_map


def class_map_to_rgb(class_map, class_colors=CLASS_COLORS):
    """
    Convert class IDs into an RGB image.
    """
    class_map = np.asarray(class_map)

    rgb = np.ones((*class_map.shape, 3), dtype=np.float32)

    for class_id, color in class_colors.items():
        rgb[class_map == class_id] = color

    return rgb


def create_error_map(
    pred_mask,
    gt_mask,
    show_class_mismatch=True
):
    """
    Create a pixel-level segmentation error map.

    Error labels:
        0 = background
        1 = correct foreground
        2 = false positive
        3 = false negative
        4 = class mismatch
    """
    pred_mask = np.asarray(pred_mask).astype(np.int64)
    gt_mask = np.asarray(gt_mask).astype(np.int64)

    if pred_mask.shape != gt_mask.shape:
        raise ValueError(
            f"Shape mismatch: prediction {pred_mask.shape}, "
            f"GT {gt_mask.shape}"
        )

    pred_fg = pred_mask > 0
    gt_fg = gt_mask > 0

    pred_class = extract_class_map(pred_mask)
    gt_class = extract_class_map(gt_mask)

    error_map = np.zeros(gt_mask.shape, dtype=np.uint8)

    # Correct foreground
    error_map[pred_fg & gt_fg] = 1

    # False positive
    error_map[pred_fg & ~gt_fg] = 2

    # False negative
    error_map[~pred_fg & gt_fg] = 3

    # Foreground overlap, but incorrect class
    if show_class_mismatch:
        class_mismatch = (
            pred_fg
            & gt_fg
            & (pred_class != gt_class)
        )
        error_map[class_mismatch] = 4

    return error_map


def calculate_instance_detection_stats(pred_mask, gt_mask):
    """
    Calculate instance-level detection statistics without IoU.

    A GT instance is counted as detected when at least one of its
    pixels overlaps with any predicted foreground pixel.

    A predicted instance is counted as matched when at least one of
    its pixels overlaps with any GT foreground pixel.

    Notes
    -----
    This measures instance presence only. It does not evaluate:

        - segmentation quality
        - overlap percentage
        - instance splitting
        - instance merging
        - one-to-one matching

    Returns
    -------
    stats : dict
    """
    pred_mask = np.asarray(pred_mask).astype(np.int64)
    gt_mask = np.asarray(gt_mask).astype(np.int64)

    if pred_mask.shape != gt_mask.shape:
        raise ValueError(
            f"Shape mismatch: prediction {pred_mask.shape}, "
            f"GT {gt_mask.shape}"
        )

    # Unique encoded instance IDs, excluding background
    pred_instance_ids = np.unique(pred_mask)
    pred_instance_ids = pred_instance_ids[pred_instance_ids > 0]

    gt_instance_ids = np.unique(gt_mask)
    gt_instance_ids = gt_instance_ids[gt_instance_ids > 0]

    pred_fg = pred_mask > 0
    gt_fg = gt_mask > 0

    # --------------------------------------------------------
    # GT instances detected by any predicted foreground overlap
    # --------------------------------------------------------
    detected_gt_ids = []
    not_detected_gt_ids = []

    for gt_id in gt_instance_ids:
        gt_instance = gt_mask == gt_id

        if np.any(pred_fg[gt_instance]):
            detected_gt_ids.append(int(gt_id))
        else:
            not_detected_gt_ids.append(int(gt_id))

    # --------------------------------------------------------
    # Predicted instances overlapping any GT foreground
    # --------------------------------------------------------
    matched_pred_ids = []
    unmatched_pred_ids = []

    for pred_id in pred_instance_ids:
        pred_instance = pred_mask == pred_id

        if np.any(gt_fg[pred_instance]):
            matched_pred_ids.append(int(pred_id))
        else:
            unmatched_pred_ids.append(int(pred_id))

    num_gt = len(gt_instance_ids)
    num_pred = len(pred_instance_ids)

    num_detected = len(detected_gt_ids)
    num_not_detected = len(not_detected_gt_ids)

    num_matched_pred = len(matched_pred_ids)
    num_unmatched_pred = len(unmatched_pred_ids)

    if num_gt > 0:
        detected_ratio = num_detected / num_gt
        not_detected_ratio = num_not_detected / num_gt
    else:
        detected_ratio = np.nan
        not_detected_ratio = np.nan

    if num_pred > 0:
        matched_pred_ratio = num_matched_pred / num_pred
        unmatched_pred_ratio = num_unmatched_pred / num_pred
    else:
        matched_pred_ratio = np.nan
        unmatched_pred_ratio = np.nan

    stats = {
        "num_gt_instances": num_gt,
        "num_pred_instances": num_pred,

        "num_detected_gt_instances": num_detected,
        "num_not_detected_gt_instances": num_not_detected,

        "detected_gt_ratio": detected_ratio,
        "not_detected_gt_ratio": not_detected_ratio,

        "num_matched_pred_instances": num_matched_pred,
        "num_unmatched_pred_instances": num_unmatched_pred,

        "matched_pred_ratio": matched_pred_ratio,
        "unmatched_pred_ratio": unmatched_pred_ratio,

        "detected_gt_ids": detected_gt_ids,
        "not_detected_gt_ids": not_detected_gt_ids,

        "matched_pred_ids": matched_pred_ids,
        "unmatched_pred_ids": unmatched_pred_ids,
    }

    return stats


def visualize_prediction_gt_error(
    raw_image,
    pred_mask,
    gt_mask,
    show_class_mismatch=True,
    save_path="raw_prediction_gt_error.png",
    dpi=300
):
    """
    Save raw image, prediction, GT and error map in one horizontal row.

    Also calculate and write instance-level detection statistics.

    Detection definition:
        A GT instance is detected when at least one pixel overlaps
        with any predicted foreground pixel.

    The figure is saved as PNG and is not displayed in JupyterLab.

    Returns
    -------
    error_map : (H, W) uint8 array

    stats : dict
        Instance-level detection statistics.
    """
    raw_image = np.asarray(raw_image)
    pred_mask = np.asarray(pred_mask).astype(np.int64)
    gt_mask = np.asarray(gt_mask).astype(np.int64)

    if pred_mask.shape != gt_mask.shape:
        raise ValueError(
            f"Prediction shape {pred_mask.shape} does not match "
            f"GT shape {gt_mask.shape}"
        )

    if raw_image.shape[:2] != gt_mask.shape:
        raise ValueError(
            f"Raw image shape {raw_image.shape[:2]} does not match "
            f"mask shape {gt_mask.shape}"
        )

    # Ensure PNG output
    if not save_path.lower().endswith(".png"):
        save_path += ".png"

    pred_class = extract_class_map(pred_mask)
    gt_class = extract_class_map(gt_mask)

    pred_rgb = class_map_to_rgb(pred_class)
    gt_rgb = class_map_to_rgb(gt_class)

    error_map = create_error_map(
        pred_mask,
        gt_mask,
        show_class_mismatch=show_class_mismatch
    )

    stats = calculate_instance_detection_stats(
        pred_mask=pred_mask,
        gt_mask=gt_mask
    )

    if show_class_mismatch:
        error_colors = [
            (0.00, 0.00, 0.00),  # background
            (0.00, 0.00, 0.00),  # correct
            (1.00, 0.55, 0.00),  # false positive
            (0.10, 0.45, 1.00),  # false negative
            (0.30, 0.30, 0.30),  # class mismatch
        ]

        error_labels = [
            "Background",
            "Correct",
            "False positive",
            "False negative",
            "Class mismatch",
        ]

    else:
        error_colors = [
            (0.00, 0.00, 0.00),  # background
            (0.00, 0.00, 0.00),  # correct
            (1.00, 0.55, 0.00),  # false positive
            (0.10, 0.45, 1.00),  # false negative
        ]

        error_labels = [
            "Background",
            "Correct",
            "False positive",
            "False negative",
        ]

    error_cmap = ListedColormap(error_colors)

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(16, 4.8)
    )

    # --------------------------------------------------------
    # Raw image
    # --------------------------------------------------------
    if raw_image.ndim == 2:
        axes[0].imshow(raw_image, cmap="gray")
    elif raw_image.ndim == 3 and raw_image.shape[-1] in (3, 4):
        axes[0].imshow(raw_image)
    else:
        raise ValueError(
            "raw_image must have shape (H, W), (H, W, 3), "
            "or (H, W, 4)."
        )

    axes[0].set_title("Raw image")

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------
    axes[1].imshow(
        pred_rgb,
        interpolation="nearest"
    )
    axes[1].set_title(
        f"Prediction\n"
        f"Instances: {stats['num_pred_instances']}"
    )

    # --------------------------------------------------------
    # Ground truth
    # --------------------------------------------------------
    axes[2].imshow(
        gt_rgb,
        interpolation="nearest"
    )
    axes[2].set_title(
        f"Ground truth\n"
        f"Instances: {stats['num_gt_instances']}"
    )

    # --------------------------------------------------------
    # Error map
    # --------------------------------------------------------
    axes[3].imshow(
        error_map,
        cmap=error_cmap,
        vmin=0,
        vmax=len(error_colors) - 1,
        interpolation="nearest"
    )
    axes[3].set_title("Error map")

    for ax in axes:
        ax.axis("off")

    # Error legend
    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="s",
            linestyle="none",
            markerfacecolor=color,
            markeredgecolor="black",
            markeredgewidth=0.5,
            markersize=8,
            label=label
        )
        for color, label in zip(
            error_colors[1:],
            error_labels[1:]
        )
    ]

    axes[3].legend(
        handles=handles,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        frameon=False,
        fontsize=8
    )

    # --------------------------------------------------------
    # Instance statistics below the four images
    # --------------------------------------------------------
    detected_ratio = stats["detected_gt_ratio"]
    not_detected_ratio = stats["not_detected_gt_ratio"]
    unmatched_pred_ratio = stats["unmatched_pred_ratio"]

    detected_text = (
        "N/A"
        if np.isnan(detected_ratio)
        else f"{detected_ratio:.1%}"
    )

    not_detected_text = (
        "N/A"
        if np.isnan(not_detected_ratio)
        else f"{not_detected_ratio:.1%}"
    )

    unmatched_pred_text = (
        "N/A"
        if np.isnan(unmatched_pred_ratio)
        else f"{unmatched_pred_ratio:.1%}"
    )

    stats_text = (
        f"GT instances: {stats['num_gt_instances']}    |    "
        f"Predicted instances: {stats['num_pred_instances']}    |    "
        f"Detected GT: {stats['num_detected_gt_instances']} "
        f"({detected_text})    |    "
        f"Not detected GT: {stats['num_not_detected_gt_instances']} "
        f"({not_detected_text})    |    "
        f"Unmatched predictions: {stats['num_unmatched_pred_instances']} "
        f"({unmatched_pred_text})"
    )

    fig.text(
        0.5,
        0.03,
        stats_text,
        ha="center",
        va="bottom",
        fontsize=9
    )

    fig.subplots_adjust(
        left=0.02,
        right=0.88,
        top=0.90,
        bottom=0.15,
        wspace=0.05
    )

    # Save only; do not display in JupyterLab
    fig.savefig(
        save_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    return error_map, stats

In [12]:
from pathlib import Path
GT_ROOT = "/home/liwa/data/datasets/yeastcity/gtFine/test/10001_10130/"
PRED_ROOT = "./predicted_v3model/"
RAW_ROOT = "/home/liwa/data/datasets/yeastcity/leftImg8bit/test/10001_10130/"

pred_list = list(Path(PRED_ROOT).rglob("*_gtFine_instanceIds.png"))
pred_list.sort()
gt_list = list(Path(GT_ROOT).rglob("*_gtFine_instanceIds.png"))
gt_list.sort()
raw_list = list(Path(RAW_ROOT).rglob("*_leftImg8bit.png"))
raw_list.sort()

In [13]:
len(pred_list)

130

In [14]:
len(gt_list)

130

In [15]:
len(raw_list)

130

In [8]:
for i in trange(0, len(raw_list)):
    raw_rgb = imread(raw_list[i])
    pred_mask = imread(pred_list[i])
    gt_mask = imread(gt_list[i])
    error_map = visualize_prediction_gt_error(
        raw_image=raw_rgb,
        pred_mask=pred_mask,
        gt_mask=gt_mask,
        save_path="./merge_check_prediction_v3/"+raw_list[i].stem[:5]+"_raw_prediction_gt_error.png",
        dpi=300
    )

100%|██████████████████████████████████████████████████████████████| 130/130 [03:25<00:00,  1.58s/it]


In [23]:
!pwd

/home/liwa/data/PytrochDeepyeast
